# Never Let Your AI App Go Down: Automatic Fallbacks with Portkey

LLM provider outages are inevitable. But when OpenAI goes down, your AI app shouldn't go down with it. In this cookbook, we'll implement automatic fallback routing that switches between providers in under 1ms, ensuring 99.99% uptime for your AI features.

## What You'll Learn
- Setting up a fallback routing strategy through the AI Gateway
- Configuring automatic fallbacks from OpenAI to Anthropic
- Testing the fallback mechanism by simulating an API failure

## Prerequisites
- Python 3.9+
- Portkey API Key ([get one free](https://app.portkey.ai))
- OpenAI API Key
- Anthropic API Key (or any secondary provider)

**Time to complete**: ~5 minutes

*Note: Portkey currently processes over 500B tokens across 125M daily requests for 24,000+ organizations. We recently raised a $15M Series A and were named a 2025 Gartner® Cool Vendor™. Reliability at scale is our core mission.*

## Step 1: Installation

First, install the Portkey SDK if you haven't already.

In [ ]:
# Install Portkey AI SDK
!pip install -q portkey-ai

## Step 2: Setup & Initialization

We initialize the Portkey client. For fallbacks, we don't need to specify the provider auth at initialization, because we will pass the Virtual Keys (or raw keys) within the routing configuration itself. 

In production, we highly recommend using Portkey Virtual Keys (secure vault keys stored in Portkey) rather than raw API keys in your config.

In [ ]:
import os
from portkey_ai import Portkey

# Set your keys here (we recommend using environment variables)
os.environ["PORTKEY_API_KEY"] = "your-portkey-api-key"
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-api-key"

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

client = Portkey(api_key=PORTKEY_API_KEY)
print("✅ Portkey client initialized!")

## Step 3: Define the Fallback Configuration

The power of Portkey lies in its configuration object. We define a simple JSON strategy that tells Portkey: "Try OpenAI first. If it fails for ANY reason (status code 4xx or 5xx, or a timeout), immediately route the exact same request to Anthropic."

In [ ]:
# Define the fallback routing strategy
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {
            "provider": "openai",
            "api_key": OPENAI_API_KEY,
            "override_params": {"model": "gpt-4o-mini"}
        },
        {
            "provider": "anthropic",
            "api_key": ANTHROPIC_API_KEY,
            "override_params": {"model": "claude-3-5-sonnet-20241022"}
        }
    ]
}

# Apply the config to our client
secure_client = client.with_options(config=fallback_config)
print("✅ Fallback configuration applied!")

## Step 4: Normal Execution (OpenAI Succeeds)

Let's make a request. Since OpenAI is up and running, Portkey will route this to `gpt-4o-mini`.

In [ ]:
try:
    response = secure_client.chat.completions.create(
        messages=[{"role": "user", "content": "What color is the sky?"}]
    )
    
    print(f"✅ Target Model Used: {response.model}")
    print(f"✅ Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")

## Step 5: Simulating an Outage (Anthropic takes over)

Let's simulate an OpenAI outage by intentionally passing an invalid OpenAI API key in our configuration. Portkey will attempt the call to OpenAI, instantly receive an authentication error, and automatically fallback to Anthropic. Your app never crashes.

In [ ]:
# Deliberately break the OpenAI configuration
broken_fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {
            "provider": "openai",
            "api_key": "sk-invalid-key-to-simulate-outage", # Intentionally broken
            "override_params": {"model": "gpt-4o-mini"}
        },
        {
            "provider": "anthropic",
            "api_key": ANTHROPIC_API_KEY,
            "override_params": {"model": "claude-3-5-sonnet-20241022"}
        }
    ]
}

# Apply the new config
resilient_client = client.with_options(config=broken_fallback_config)

try:
    print("📡 Attempting request... (OpenAI will fail, watch Claude take over)")
    response = resilient_client.chat.completions.create(
        messages=[{"role": "user", "content": "Who are you?"}]
    )
    
    print("\n🛡️ Request Succeeded despite OpenAI failure!")
    # Note how the 'model' returned is Anthropic's Claude!
    print(f"✅ Target Model Used: {response.model}")
    print(f"✅ Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Fatal Error: {e}")
    print("Did you forget to set the Anthropic API key?")

## 🎯 Key Takeaways

1. **Zero-Code Reliability**: Adding fallbacks requires zero changes to your core application logic or prompt structures. You simply wrap your client with a configuration JSON.
2. **Instant Failover**: When OpenAI fails, Portkey routes the exact same request body to Anthropic in under 1ms. Your end-users never notice.
3. **Model Translation**: Portkey automatically translates standard OpenAI-formatted messages into Anthropic's required format behind the scenes.
4. **Complete Tracing**: When you view this request in the Portkey Observability Dashboard, you'll see the exact waterfall of events: OpenAI (Failed) → Anthropic (Success).

## Common Gotchas
- Be aware of capability mismatches. Don't fallback from a massive reasoning model (GPT-4o) to a tiny fast model (Claude Haiku) unless your prompt can tolerate the quality drop.
- You can nest strategies! You can load-balance across three OpenAI keys, and if *all* of them fail, fallback to a load-balanced pool of Anthropic keys.

## 🚀 Next Steps

- **Load Balancing**: [Distribute traffic across multiple API keys](https://portkey.ai/docs/product/ai-gateway/load-balancing)
- **Enable Caching**: [Reduce costs with semantic caching](https://portkey.ai/docs/product/ai-gateway/cache-simple-and-semantic)
- **Add Guardrails**: [Protect your LLM outputs](https://portkey.ai/docs/product/guardrails)
- **Monitor in Dashboard**: [Analyze logs with custom metadata](https://portkey.ai/docs/product/observability)

---

⭐ **Star our gateway**: [github.com/Portkey-AI/gateway](https://github.com/Portkey-AI/gateway)

📚 **Full docs**: [portkey.ai/docs](https://portkey.ai/docs/)

🐦 **Follow us**: [@PortkeyAI](https://x.com/PortkeyAI)
